# Split-CIFAR-10: Incremental Learning with NWF

5 tasks, 2 classes each. Train ConvVAE once, add charges per task. No catastrophic forgetting.

In [ ]:
import numpy as np
from torchvision.datasets import CIFAR10
from torchvision.transforms import ToTensor
from nwf import Charge, Field
from nwf.vision import ConvVAEEncoder

In [ ]:
ds = CIFAR10(root='./data', train=True, download=True, transform=ToTensor())
X = np.stack([ds[i][0].numpy() for i in range(len(ds))])
y = np.array([ds[i][1] for i in range(len(ds))])

n_tasks = 5
tasks = []
for t in range(n_tasks):
    c_start, c_end = t * 2, (t + 1) * 2
    mask = (y >= c_start) & (y < c_end)
    tasks.append((X[mask], y[mask]))

X_full = np.concatenate([t[0] for t in tasks])
y_full = np.concatenate([t[1] for t in tasks])

In [ ]:
print('Training ConvVAE...')
enc = ConvVAEEncoder(input_shape=(3, 32, 32), latent_dim=64)
enc.fit(X_full, epochs=5, batch_size=128)

In [ ]:
def compute_class_charges(enc, X, y):
    z_all, sigma_all = enc.encode(X)
    if z_all.ndim == 1:
        z_all = z_all.reshape(1, -1)
        sigma_all = sigma_all.reshape(1, -1)
    classes = np.unique(y)
    z_means = [np.mean(z_all[y == c], axis=0) for c in classes]
    sigma_means = [np.mean(sigma_all[y == c], axis=0) + 1e-6 for c in classes]
    return np.array(z_means), np.array(sigma_means)

In [ ]:
field = Field()
accuracies = []
k = 5

for task_id, (X_t, y_t) in enumerate(tasks):
    z_mean, sigma_mean = compute_class_charges(enc, X_t, y_t)
    class_ids = np.unique(y_t)
    for i, c in enumerate(class_ids):
        ch = Charge(z=z_mean[i].astype(np.float64), sigma=sigma_mean[i].astype(np.float64))
        field.add(ch, labels=[int(c)], ids=[task_id * 10 + c])

    n_correct, n_total = 0, 0
    for t in range(task_id + 1):
        X_te, y_te = tasks[t][0], tasks[t][1]
        z_te, s_te = enc.encode(X_te[:100])
        if z_te.ndim == 1:
            z_te = z_te.reshape(1, -1)
            s_te = s_te.reshape(1, -1)
        for i in range(len(z_te)):
            q = Charge(z=z_te[i], sigma=s_te[i])
            _, _, lab = field.search(q, k=k)
            votes = np.bincount(np.array(lab[0]).astype(int), minlength=10)
            pred = int(np.argmax(votes))
            if pred == y_te[i]:
                n_correct += 1
            n_total += 1
    acc = n_correct / n_total if n_total > 0 else 0
    accuracies.append(acc)
    print(f'After task {task_id + 1}: accuracy = {acc:.4f}')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(accuracies) + 1), accuracies, 'o-')
plt.xlabel('Task'); plt.ylabel('Accuracy'); plt.title('Split-CIFAR-10')
plt.xticks(range(1, len(accuracies) + 1)); plt.show()

Run full script: `python examples/split_cifar.py --epochs 5`